In [1]:
import os
print(os.getcwd())

C:\Users\chime\OneDrive\Desktop\aadhaar_accessibility_analyzer\notebooks


In [4]:
print(os.listdir("../data/raw"))

['api_data_aadhar_enrolment_0_500000.csv', 'api_data_aadhar_enrolment_1000000_1006029.csv', 'api_data_aadhar_enrolment_500000_1000000.csv']


In [7]:
import pandas as pd
import glob
files = glob.glob("../data/raw/*csv")
df_list = []
for file in files:
    df = pd.read_csv(file)
    df_list.append(df)
df_enroll = pd.concat(df_list,ignore_index = True)

In [8]:
df_enroll.shape

(1006029, 7)

In [9]:
df_enroll.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,Meghalaya,East Khasi Hills,793121,11,61,37
1,09-03-2025,Karnataka,Bengaluru Urban,560043,14,33,39
2,09-03-2025,Uttar Pradesh,Kanpur Nagar,208001,29,82,12
3,09-03-2025,Uttar Pradesh,Aligarh,202133,62,29,15
4,09-03-2025,Karnataka,Bengaluru Urban,560016,14,16,21


In [11]:
df_enroll.to_csv("../data/raw/enrollment_full.csv",index = False)

In [2]:
import pandas as pd
import os
path = "../data/raw"
demo_files = [f for f in os.listdir(path) if "demographic" in f]
print(demo_files)

['api_data_aadhar_demographic_0_500000.csv', 'api_data_aadhar_demographic_1000000_1500000.csv', 'api_data_aadhar_demographic_1500000_2000000.csv', 'api_data_aadhar_demographic_2000000_2071700.csv', 'api_data_aadhar_demographic_500000_1000000.csv']


In [4]:
import pandas as pd
df_list = []
for file in demo_files:
    df = pd.read_csv(os.path.join(path,file))
    df_list.append(df)
df_demo = pd.concat(df_list,ignore_index = True)

In [5]:
df_demo.shape
df_demo.head()

,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,Uttar Pradesh,Gorakhpur,273213,49,529
1,01-03-2025,Andhra Pradesh,Chittoor,517132,22,375
2,01-03-2025,Gujarat,Rajkot,360006,65,765
3,01-03-2025,Andhra Pradesh,Srikakulam,532484,24,314
4,01-03-2025,Rajasthan,Udaipur,313801,45,785


In [6]:
df_demo.to_csv("../data/raw/demographic_full.csv",index = False)

In [8]:
import pandas as pd
import os
path = "../data/raw"
bio_files = [f for f in os.listdir(path) if "biometric" in f]
print(bio_files)

['api_data_aadhar_biometric_0_500000.csv', 'api_data_aadhar_biometric_1000000_1500000.csv', 'api_data_aadhar_biometric_1500000_1861108.csv', 'api_data_aadhar_biometric_500000_1000000.csv']


In [9]:
import pandas as pd
df_list = []
for file in bio_files:
    df = pd.read_csv(os.path.join(path,file))
    df_list.append(df)
df_bio = pd.concat(df_list,ignore_index = True)

In [11]:
df_bio.shape
df_bio.head()

,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu and Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [12]:
df_bio.to_csv("../data/raw/biometric_full.csv",index = False)

## cleaning and merging

In [1]:
import pandas as pd
df_enroll = pd.read_csv("../data/raw/enrollment_full.csv")
df_demo = pd.read_csv("../data/raw/demographic_full.csv")
df_bio  = pd.read_csv("../data/raw/biometric_full.csv")

In [2]:
df_enroll.columns = df_enroll.columns.str.strip().str.lower()
df_demo.columns = df_demo.columns.str.strip().str.lower()
df_bio.columns = df_bio.columns.str.strip().str.lower()

In [3]:
df_enroll = df_enroll.fillna(0)
df_demo = df_demo.fillna(0)
df_bio = df_bio.fillna(0)

In [4]:
df_enroll = df_enroll.drop_duplicates()
df_demo = df_demo.drop_duplicates()
df_bio = df_bio.drop_duplicates()

In [5]:
df_merge1 = pd.merge(
    df_enroll,
    df_demo,
    on=["date", "state", "district", "pincode"],
    how="inner"
)
df_final = pd.merge(
    df_merge1,
    df_bio,
    on=["date", "state", "district", "pincode"],
    how="inner"
)

In [6]:
df_final.shape

(544834, 11)

In [7]:
df_final.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_
0,01-04-2025,Punjab,Ludhiana,141007,374,110,25,206,2000,925,1404
1,01-04-2025,Madhya Pradesh,Ashok Nagar,473335,125,29,22,181,975,1418,587
2,01-04-2025,Jharkhand,Ranchi,834001,187,174,21,210,1855,1494,2283
3,01-04-2025,Gujarat,Surat,394107,169,69,32,206,2239,608,717
4,01-04-2025,Karnataka,Bengaluru,560036,204,34,29,131,781,314,192


In [8]:
df_final.to_csv("../data/processed/final_merged_data.csv",index = False)